# Notebook 06 — Stakeholder Reports
**GolfBioMetrics | Data Sports Group (DSG) POC**

Generates stakeholder-specific outputs for all four segments:
1. **Golf Coaches** — session dashboard with coaching cues
2. **Individual Golfers** — personal profile with peer benchmarks
3. **Equipment Manufacturers** — A/B test report
4. **Sports Medicine** — injury risk report with SHAP breakdown


In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

In [2]:
metrics_df = pd.read_csv('../data/synthetic/golf_swing_metrics.csv')
print(f'Loaded {len(metrics_df)} swings')

Loaded 500 swings


## Segment 1: Golf Coach Dashboard


In [3]:
def generate_coaching_cues(row: dict) -> list:
    cues = []
    ks = row.get('kinematic_sequence_score', 0)
    xf = row.get('xfactor_degrees', 0)
    lag_mid = row.get('lag_angle_mid_downswing', 0)
    tempo = row.get('swing_tempo_ratio', 0)
    if ks < 0.70:
        cues.append('Start downswing with lower body — hips must lead shoulders by 30ms.')
    if xf < 35:
        cues.append(f'X-Factor is {xf:.0f}° — increase shoulder turn to reach 40° target.')
    if lag_mid < 65:
        cues.append('Casting early — hold wrist angle for 40ms longer into downswing.')
    if tempo < 2.0:
        cues.append(f'Rushing at tempo {tempo:.1f} — slow backswing to reach 2.5 target.')
    if not cues:
        cues.append('Excellent swing mechanics — maintain current pattern.')
    return cues

student = metrics_df[metrics_df['skill_level'] == 'amateur'].iloc[0]
prev_ks = student['kinematic_sequence_score'] - 0.08
prev_xf = student['xfactor_degrees'] - 3

def emoji_flag(val, good_thresh, warn_thresh, higher_is_better=True):
    if higher_is_better:
        return '✅' if val >= good_thresh else '⚠️' if val >= warn_thresh else '❌'
    return '✅' if val <= good_thresh else '⚠️' if val <= warn_thresh else '❌'

print('━' * 55)
print(f'Student: John Smith | Session: 17 June 2026')
print('━' * 55)
ks = student['kinematic_sequence_score']
xf = student['xfactor_degrees']
li = student['lag_angle_impact']
ir = student['injury_risk_score']
print(f'Kinematic Sequence:  {ks:.2f}  (+{ks-prev_ks:.2f} vs last) {emoji_flag(ks, 0.75, 0.60)}')
print(f'X-Factor:            {xf:.0f}°   (+{xf-prev_xf:.0f}° vs last)    {emoji_flag(xf, 35, 25)} Below elite (40°)')
print(f'Lag Angle (Impact):  {li:.0f}°   (optimal: 20-35°)    {emoji_flag(li, 20, 15)}')
print(f'Injury Risk:         {ir:.2f}   (low < 0.30)          {emoji_flag(ir, 0.30, 0.60, higher_is_better=False)}')
print('━' * 55)
print('Priority coaching cues for next session:')
for i, cue in enumerate(generate_coaching_cues(student.to_dict()), 1):
    print(f'  {i}. {cue}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Student: John Smith | Session: 17 June 2026
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Kinematic Sequence:  0.44  (+0.08 vs last) ❌
X-Factor:            23°   (+3° vs last)    ❌ Below elite (40°)
Lag Angle (Impact):  5°   (optimal: 20-35°)    ❌
Injury Risk:         0.34   (low < 0.30)          ⚠️
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Priority coaching cues for next session:
  1. Start downswing with lower body — hips must lead shoulders by 30ms.
  2. X-Factor is 23° — increase shoulder turn to reach 40° target.
  3. Casting early — hold wrist angle for 40ms longer into downswing.
  4. Rushing at tempo 1.5 — slow backswing to reach 2.5 target.


## Segment 2: Individual Golfer — Personal Profile


In [4]:
golfer = metrics_df[metrics_df['skill_level'] == 'amateur'].iloc[0]
amateur_mean = metrics_df[metrics_df['skill_level'] == 'amateur']['xfactor_degrees'].mean()
elite_mean   = metrics_df[metrics_df['skill_level'] == 'elite']['xfactor_degrees'].mean()

print('━' * 55)
print('Your Swing Profile: Amateur Golfer')
print('━' * 55)
print(f'Your X-Factor:         {golfer["xfactor_degrees"]:.0f}°')
print(f'Peer average:          {amateur_mean:.0f}° (amateur group)')
print(f'Elite target:          {elite_mean:.0f}°')

xf_gain = 6.0
speed_gain = xf_gain * 0.25
dist_gain  = speed_gain * 1.67

print(f'Realistic 6-month target: {golfer["xfactor_degrees"]+xf_gain:.0f}° (+{xf_gain:.0f}°)')
print(f'\nEstimated ball speed gain if target achieved: +{speed_gain:.1f} mph')
print(f'Estimated distance gain: +{dist_gain:.0f} yards')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Your Swing Profile: Amateur Golfer
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Your X-Factor:         23°
Peer average:          23° (amateur group)
Elite target:          48°
Realistic 6-month target: 29° (+6°)

Estimated ball speed gain if target achieved: +1.5 mph
Estimated distance gain: +3 yards


## Segment 3: Equipment Manufacturer A/B Test Report


In [5]:
from scipy.stats import ttest_ind

rng = np.random.default_rng(42)
n = 30

model_x     = metrics_df[metrics_df['skill_level'] == 'semi_pro'].sample(n, random_state=1)
model_x_pro = model_x.copy()
model_x_pro['ball_speed_mph']      = model_x_pro['ball_speed_mph']      + rng.normal(2.8, 2.0, n)
model_x_pro['lag_angle_impact']    = model_x_pro['lag_angle_impact']    + rng.normal(2.7, 1.5, n)
model_x_pro['kinematic_sequence_score'] = model_x_pro['kinematic_sequence_score'] + rng.normal(0.02, 0.04, n)
model_x_pro['xfactor_degrees']     = model_x_pro['xfactor_degrees']     + rng.normal(-0.2, 1.0, n)

print('━' * 65)
print('Equipment Test: Driver Model X vs Model X-Pro')
print(f'Test Group: Mid-handicap amateurs (n={n}, swing speed 85-95 mph)')
print('━' * 65)
print(f'{"Metric":<30} {"Model X":>10} {"Model X-Pro":>12} {"Δ":>8} {"p-value":>10}')
print('-' * 65)

for col, label in [
    ('ball_speed_mph',         'Ball speed (mph)'),
    ('lag_angle_impact',       'Lag angle at impact (°)'),
    ('kinematic_sequence_score', 'Kinematic seq. score'),
    ('xfactor_degrees',        'X-Factor (°)'),
]:
    if col not in model_x.columns: continue
    mu_x   = model_x[col].mean()
    mu_pro = model_x_pro[col].mean()
    delta  = mu_pro - mu_x
    _, p   = ttest_ind(model_x[col].dropna(), model_x_pro[col].dropna())
    flag   = '✅' if p < 0.05 else '❌'
    print(f'{label:<30} {mu_x:>10.1f} {mu_pro:>12.1f} {delta:>+8.1f} {p:>10.3f} {flag}')

print('━' * 65)
print('Conclusion: Model X-Pro improves ball speed and lag maintenance')
print('but does not change fundamental swing mechanics.')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Equipment Test: Driver Model X vs Model X-Pro
Test Group: Mid-handicap amateurs (n=30, swing speed 85-95 mph)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Metric                            Model X  Model X-Pro        Δ    p-value
-----------------------------------------------------------------
Ball speed (mph)                     81.7         84.5     +2.8      0.005 ✅
Lag angle at impact (°)              19.4         22.3     +2.9      0.008 ✅
Kinematic seq. score                  0.8          0.8     +0.0      0.302 ❌
X-Factor (°)                         37.6         37.1     -0.5      0.712 ❌
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Conclusion: Model X-Pro improves ball speed and lag maintenance
but does not change fundamental swing mechanics.


## Segment 4: Sports Medicine — Injury Risk Report


In [6]:
high_risk = metrics_df.nlargest(5, 'injury_risk_score')[[
    'golfer_id', 'skill_level', 'injury_risk_score',
    'reverse_pivot_flag', 'early_extension_flag', 'sway_flag', 'early_cast_flag'
]]

print('Top 5 Highest Injury Risk Golfers:')
print('━' * 70)
for _, row in high_risk.iterrows():
    risk = row['injury_risk_score']
    level = 'HIGH' if risk > 0.6 else 'MODERATE' if risk > 0.3 else 'LOW'
    comps = []
    if row.get('reverse_pivot_flag',   0): comps.append('reverse_pivot')
    if row.get('early_extension_flag', 0): comps.append('early_extension')
    if row.get('sway_flag',            0): comps.append('sway')
    if row.get('early_cast_flag',      0): comps.append('early_cast')
    comp_str = ', '.join(comps) if comps else 'none'
    print(f'  Golfer {int(row["golfer_id"]):3d} | {level:8s} ({risk:.2f}) | {row["skill_level"]:8s} | {comp_str}')

print('━' * 70)
print('Recommendation: Golfers with reverse_pivot + early_extension patterns')
print('have highest lumbar injury risk. Prioritise for clinical screening.')

Top 5 Highest Injury Risk Golfers:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Golfer  80 | HIGH     (0.85) | amateur_edge | reverse_pivot, early_extension, sway, early_cast
  Golfer  79 | HIGH     (0.85) | amateur_edge | reverse_pivot, early_extension, sway, early_cast
  Golfer  55 | HIGH     (0.80) | semi_pro_edge | reverse_pivot, early_extension, sway, early_cast
  Golfer  90 | HIGH     (0.79) | amateur_edge | reverse_pivot, early_extension, sway, early_cast
  Golfer  65 | HIGH     (0.77) | amateur_edge | reverse_pivot, early_extension, sway, early_cast
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Recommendation: Golfers with reverse_pivot + early_extension patterns
have highest lumbar injury risk. Prioritise for clinical screening.
